Mapeamento Regional e Temporal: Identificar quais regiões e estados do Brasil apresentam as maiores taxas de crimes contra mulheres e a população LGBTQIA+, bem como analisar se há uma tendência de aumento ou redução desses índices ao longo dos últimos anos.


Avaliação de Políticas Públicas: Avaliar, através de dados, se a implementação de leis de proteção específicas resultou em impactos reais na diminuição das taxas de criminalidade nas regiões analisadas.

Análise de Tipificação: Investigar a qualidade dos dados oficiais, buscando indícios de falhas de tipificação (como registrar crimes de ódio como crimes comuns).

Implicação Prática: Fornecer um panorama analítico que possa auxiliar gestores de segurança e organizações da sociedade civil na alocação de recursos (como delegacias especializadas) e no aprimoramento dos sistemas de registro de boletins de ocorrência.

# Datas:

**Início da criminalização de crimes contra LGBTQ**:

Desde junho de 2019, o STF, na ADO nº 26, criminalizou a LGBTfobia no Brasil, enquadrando atos de ódio e discriminação contra orientação sexual ou identidade de gênero na Lei de Racismo (Lei 7.716/1989). A conduta é crime inafiançável e imprescritível, com penas de 1 a 3 anos de prisão, podendo ser maior em casos de injúria racial. 
Homicídio Qualificado/Hediondo: A discriminação que motiva um homicídio (LGBTIcídio) torna o crime hediondo, com pena maior (12 a 30 anos).

**Lei do Feminicídio**:

A Lei do Feminicídio (Lei nº 13.104/2015), atualizada pela Lei nº 14.994/2024, torna o assassinato de mulheres por razões de gênero um crime autônomo e hediondo, com penas severas de 20 a 40 anos de reclusão. Considera-se feminicídio quando envolve violência doméstica/familiar ou menosprezo/discriminação à condição de mulher, com agravantes para crimes cometidos na gravidez, contra menores de 14 ou maiores de 60 anos, ou na presença de familiares.

# Perguntas

Quais regiões e estados do Brasil concentram as maiores taxas de violência contra mulheres e pessoas LGBTQIA+, e existe padrão de sazonalidade ou escalonada ao longo dos anos?

Há uma correlação observável entre a implementação de marcos legais de proteção (como a Lei do Feminicídio ou a criminalização da homofobia pelo STF) e a diminuição das ocorrências em estados específicos?

A proporção de feminicídios em relação ao total de homicídios de mulheres indica uma subnotificação ou falha na tipificação do crime em determinadas bases de dados estaduais?

Como a ausência de campos específicos para orientação sexual e identidade de gênero nos registros oficiais impacta a dimensão dos dados quando comparados aos levantamentos de ONGs e observatórios independentes?

### Todos os plots de graficos feitos com a lib plotly foram gerados pelo codex e gemini

In [1]:
from pathlib import Path
from dotenv import dotenv_values

import requests

import pandas as pd
import plotly.express as px

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent

from sqlalchemy import (
    BigInteger,
    Boolean,
    Column,
    DateTime,
    Float,
    MetaData,
    String,
    Table,
    Text,
    create_engine,
    inspect,
)
from sqlalchemy.engine import URL
from sqlalchemy.schema import CreateSchema
from sqlalchemy import Integer, MetaData, Table, func, select


O codigo a baixo foi comentado, pois a construção do banco de dados com as tabelas da SINESP é muito custosa tendo em vista que deixaremos de utilizar esses dados, mas o mantive aqui para referência futura

In [2]:
# def build_postgres_engine(host: str, port: int, database: str, user: str, password: str):
#     return create_engine(
#         URL.create(
#             "postgresql+psycopg2",
#             username=user,
#             password=password,
#             host=host,
#             port=port,
#             database=database,
#         )
#     )


In [3]:
# env = dotenv_values(project_root / ".env")
# try:
#     host = "localhost"
#     port = int(env.get("POSTGRES_PORT", 5432))# type:ignore
#     database = env["POSTGRES_DB"]
#     user = env["POSTGRES_USER"]
#     password = env["POSTGRES_PASSWORD"]
#     schema_name = "public"
#     table_name = "pubseg"
# except:
#     raise

# SQL

In [4]:
# engine = build_postgres_engine(host, port, database, user, password)
# metadata = MetaData(schema=schema_name)
# pubseg = Table(table_name, metadata, autoload_with=engine, schema=schema_name)

### Tabela por quantidade de ocorrências ao ano

In [5]:
# ano = func.extract("year", pubseg.c.data_referencia).cast(Integer)
# evento = func.trim(pubseg.c.evento)

# evento_counts_stmt = (
#     select(
#         evento.label("evento"),
#         ano.label("ano"),
#         func.count().label("quantidade"),
#     )
#     .where(pubseg.c.evento.is_not(None), evento != "")
#     .group_by(evento, ano)
#     .order_by(evento, ano)
# )

# evento_counts_por_ano = pd.read_sql_query(evento_counts_stmt, engine)
# evento_counts_por_ano = (
#     evento_counts_por_ano
#     .pivot(index="evento", columns="ano", values="quantidade")
#     .fillna(0)
#     .astype(int)
#     .reset_index()
# )
# evento_counts_por_ano.columns.name = None
# evento_counts_por_ano

# Sem SQL

Leitura das tabelas da SINESP a fim de agrupar e permitir a vizualização da evolução temporal da quantidade de cada ocorrência ao ano

In [ ]:
anos_pubseg = list(range(2015, 2026))
download_path = project_root / "data" /"cache"/ "xlsx_dowloads"

pubseg_frames = []
for ano in anos_pubseg:
    caminho_arquivo = pubseg_download_path / f"pubseg-{ano}.xlsx"
    db_ano = pd.read_excel(caminho_arquivo, usecols=["evento", "data_referencia"])
    db_ano["evento"] = db_ano["evento"].astype("string").str.strip()
    db_ano["ano"] = pd.to_datetime(db_ano["data_referencia"], errors="coerce").dt.year
    pubseg_frames.append(db_ano[["evento", "ano"]])

pubseg_xlsx_eventos_df = pd.concat(pubseg_frames, ignore_index=True)
pubseg_xlsx_eventos_df = pubseg_xlsx_eventos_df.dropna(subset=["evento", "ano"])
pubseg_xlsx_eventos_df = pubseg_xlsx_eventos_df.loc[pubseg_xlsx_eventos_df["evento"] != ""]

total_aparicoes_por_evento_ano_xlsx = (
    pubseg_xlsx_eventos_df
    .groupby(["evento", "ano"], as_index=False)
    .size()
    .rename(columns={"size": "total_aparicoes"})
)

total_aparicoes_por_evento_ano_xlsx_tabela = (
    total_aparicoes_por_evento_ano_xlsx
    .pivot(index="evento", columns="ano", values="total_aparicoes")
    .fillna(0)
    .astype(int)
    .reset_index()
)
total_aparicoes_por_evento_ano_xlsx_tabela.columns.name = None
total_aparicoes_por_evento_ano_xlsx_tabela

,evento,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Apreensão de Cocaína,636,648,648,648,648,648,648,648,648,648,648
1,Apreensão de Maconha,636,648,648,648,648,648,648,648,648,648,648
2,Arma de Fogo Apreendida,5724,5832,5832,5832,5832,5832,5832,5832,5832,5832,5832
3,Atendimento pré-hospitalar,324,324,324,324,324,324,324,324,324,324,324
4,Busca e salvamento,324,324,324,324,324,324,324,324,324,324,324
5,Combate a incêndios,324,324,324,324,324,324,324,324,324,324,324
6,Emissão de Alvarás de licença,324,324,324,324,324,324,324,324,324,324,324
7,Estupro,324,324,324,324,324,324,324,324,324,324,324
8,Estupro de vulnerável,324,324,324,324,324,324,324,324,324,324,324
9,Feminicídio,66600,67548,67548,67548,67560,67560,67560,67548,67548,67560,67558


Aqui foi feita uma seleção de alguns eventos de segurança publica que gostariamos de vizualiar e foram plotados a cada ano, foi esse grafico que inicialmente gerou a suspeita de falha na integridade dos dados

In [8]:
eventos_para_plotar = [
    "Estupro",
    "Estupro de vulnerável",
    "Feminicídio",
    "Homicídio doloso",
    "Lesão corporal seguida de morte",
    "Tentativa de feminicídio",
    "Tentativa de homicídio",
]

eventos_por_ano_df = (
    total_aparicoes_por_evento_ano_xlsx_tabela
    .loc[total_aparicoes_por_evento_ano_xlsx_tabela["evento"].isin(eventos_para_plotar)]
    .melt(id_vars="evento", var_name="ano", value_name="quantidade")
)

eventos_por_ano_df["ano"] = eventos_por_ano_df["ano"].astype(int)

fig = px.bar(
    eventos_por_ano_df,
    x="ano",
    y="quantidade",
    color="evento",
    barmode="group",
    title="Ocorrências por ano para eventos selecionados",
    labels={"ano": "Ano", "quantidade": "Ocorrências", "evento": "Evento"},
)
fig.show()


Para confirmar a possivel falha na integridade dos dados, fizemos uma avaliação da duplicidade de inserção, eu comparei linha a linha desconsiderando id e ano (dia e mes ainda considerado) tabela a tabela

In [9]:
from collections import Counter

anos_pubseg = list(range(2015, 2026))
pubseg_download_path = project_root / "data" / "xlsx_dowloads"

pubseg_bruto_por_ano = {
    ano: pd.read_excel(pubseg_download_path / f"pubseg-{ano}.xlsx")
    for ano in anos_pubseg
}

colunas_para_ignorar = {"id", "id_tabela"}
colunas_comparacao = sorted(
    set().union(*(set(df.columns) for df in pubseg_bruto_por_ano.values())) - colunas_para_ignorar
)

def normalizar_pubseg_para_comparacao(df: pd.DataFrame) -> pd.DataFrame:
    db = df.reindex(columns=colunas_comparacao).copy()

    if "data_referencia" in db.columns:
        data_referencia = pd.to_datetime(db["data_referencia"], errors="coerce")
        db["data_referencia"] = data_referencia.dt.strftime("%m-%d")

    for coluna in db.columns:
        serie = db[coluna]
        if pd.api.types.is_datetime64_any_dtype(serie):
            db[coluna] = pd.to_datetime(serie, errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
        elif pd.api.types.is_object_dtype(serie) or pd.api.types.is_string_dtype(serie):
            db[coluna] = serie.astype("string").str.strip()
        else:
            db[coluna] = serie.astype("string")

    return db.fillna("<NA>")

contadores_por_ano = {}
totais_por_ano = {}
for ano, db in pubseg_bruto_por_ano.items():
    db_normalizado = normalizar_pubseg_para_comparacao(db)
    linhas_normalizadas = list(db_normalizado.itertuples(index=False, name=None))
    contadores_por_ano[ano] = Counter(linhas_normalizadas)
    totais_por_ano[ano] = len(linhas_normalizadas)

matriz_sobreposicao_pubseg = pd.DataFrame(index=anos_pubseg, columns=anos_pubseg, dtype=float)

for ano_a in anos_pubseg:
    total_linhas_a = totais_por_ano[ano_a]
    contador_a = contadores_por_ano[ano_a]

    for ano_b in anos_pubseg:
        contador_b = contadores_por_ano[ano_b]
        linhas_iguais = sum(
            min(quantidade_a, contador_b.get(linha, 0))
            for linha, quantidade_a in contador_a.items()
        )
        percentual = 100 * linhas_iguais / total_linhas_a if total_linhas_a else 0
        matriz_sobreposicao_pubseg.loc[ano_a, ano_b] = percentual

matriz_sobreposicao_pubseg = matriz_sobreposicao_pubseg.round(2)
matriz_sobreposicao_pubseg



,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
2015,100.00,81.23,67.51,65.81,61.75,60.63,58.93,58.12,57.70,57.32,57.20
2016,78.68,100.00,68.71,67.15,63.81,62.45,60.68,59.90,58.91,58.45,58.31
2017,72.42,76.09,100.00,81.62,77.53,76.03,74.03,73.18,72.66,72.15,72.06
2018,70.64,74.41,81.66,100.00,79.90,78.60,76.38,75.38,74.94,74.42,74.25
2019,66.24,70.66,77.52,79.85,100.00,82.41,80.45,79.30,78.30,77.80,77.71
2020,65.04,69.15,76.02,78.55,82.41,100.00,81.65,80.46,79.41,78.94,78.81
2021,63.21,67.19,74.02,76.33,80.45,81.65,100.00,81.78,80.94,80.35,79.76
2022,62.35,66.34,73.18,75.34,79.31,80.47,81.79,100.00,81.28,80.71,80.34
2023,61.90,65.25,72.66,74.90,78.31,79.42,80.95,81.28,100.00,80.99,80.38
2024,61.48,64.72,72.14,74.37,77.80,78.94,80.35,80.70,80.98,100.00,80.85


plot  feito pelo codex apenas para tornar mais visual a comparação de duplicidade

In [10]:
fig = px.imshow(
    matriz_sobreposicao_pubseg,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="Blues",
    labels={
        "x": "Ano B",
        "y": "Ano A",
        "color": "% de linhas de A presentes em B",
    },
    title="Sobreposição percentual entre arquivos pubseg-{ano}.xlsx",
)
fig.update_xaxes(side="top")
fig.show()



# Dados Disk 100

In [12]:
csv_download_path = project_root / "data" / "csv_dowloads"
csv_paths = sorted(csv_download_path.glob("*.csv"))

csv_frames = [
    pd.read_csv(csv_path, sep=";", encoding="latin1", low_memory=False)
    for csv_path in csv_paths
]

disk100_unificado_df = pd.concat(csv_frames, ignore_index=True)



Ajuste do tipo de encoding da tabela e do tipo de Data_de_cadastro

In [13]:
def corrigir_latin1_para_utf8(valor):
    if pd.isna(valor) or not isinstance(valor, str):
        return valor
    try:
        return valor.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return valor

colunas_texto_disk100 = disk100_unificado_df.select_dtypes(include=["object", "string"]).columns

disk100_unificado_df = disk100_unificado_df.copy()
for coluna in colunas_texto_disk100:
    disk100_unificado_df[coluna] = disk100_unificado_df[coluna].map(corrigir_latin1_para_utf8)

colunas_corrigidas_disk100 = [
    corrigir_latin1_para_utf8(coluna).replace("ï»¿", "") if isinstance(coluna, str) else coluna
    for coluna in disk100_unificado_df.columns
]

disk100_unificado_df = disk100_unificado_df.copy()
disk100_unificado_df.columns = colunas_corrigidas_disk100

disk100_unificado_df = disk100_unificado_df.copy()
disk100_unificado_df["Data_de_cadastro"] = pd.to_datetime(
    disk100_unificado_df["Data_de_cadastro"],
    errors="coerce"
)


Print das colunas e tipos das informações que carregam

In [14]:

print(disk100_unificado_df.dtypes.to_string())


hash                                                        str
Data_de_cadastro                                  datetime64[us]
Canal_de_atendimento                                         str
Denúncia_emergencial                                         str
Denunciante                                                  str
Cenário_da_violação                                          str
País                                                         str
UF                                                           str
Município                                                    str
Frequência                                                   str
Início_das_violações                                         str
sl_quantidade_vitimas                                      int64
Grupo_vulnerável                                             str
Motivação                                                    str
Relação_vítima_suspeito                                      str
sl_vitima_cadastro        

Contagem de inserção de grupo vulnerável a cada mês do dataframe

In [16]:
grupo_vulneravel_mes_a_mes = (
    disk100_unificado_df.assign(
        Grupo_vulnerável=disk100_unificado_df["Grupo_vulnerável"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        mes=disk100_unificado_df["Data_de_cadastro"].dt.to_period("M").astype("string"),
    )
    .dropna(subset=["Data_de_cadastro"])
    .groupby(["Grupo_vulnerável", "mes"])
    .size()
    .unstack(fill_value=0)
    .sort_index(axis=1)
    .reset_index()
)

grupo_vulneravel_mes_a_mes


mes,Grupo_vulnerável,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
0,VIOLÊNCIA CONTRA A MULHER,31422,27755,31581,31802,32848,29020,32400,32374,35247,38254,34925,33207
1,"VIOLÊNCIA CONTRA CIDADÃO, FAMÍLIA OU COMUNIDADE",37535,33630,31451,34444,33830,30209,34030,31938,36374,38745,40891,36646
2,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,147057,147718,147393,146178,153255,134051,144306,171457,177990,171500,147512,131441
3,VIOLÊNCIA CONTRA PESSOA COM DEFICIÊNCIA,59582,56735,57711,58725,58632,56283,64867,67823,73132,75572,66721,61410
4,VIOLÊNCIA CONTRA PESSOA EM RESTRIÇÃO DE LIBERDADE,6632,4633,3638,5768,4545,4000,4918,4028,6275,5978,9396,7424
5,VIOLÊNCIA CONTRA PESSOA EM SITUAÇÃO DE RUA,2087,1785,1710,1691,2373,2139,2234,2722,2068,2407,1926,1709
6,VIOLÊNCIA CONTRA PESSOA IDOSA,93022,83777,81212,79989,80571,79042,93270,95264,103215,103906,87221,78976
7,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,4237,2949,3617,3866,4299,4281,4843,4613,4483,5647,5091,5312
8,nulo,0,0,1,0,0,0,0,0,0,0,0,0


Plot gerado pelo codex para vizualização dos grupos de interesse

In [17]:
grupos_destacados = {
    "VIOLÊNCIA CONTRA A MULHER",
    "VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+",
}

grupo_vulneravel_mes_plot = (
    disk100_unificado_df.assign(
        Grupo_vulnerável=disk100_unificado_df["Grupo_vulnerável"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
    )
    .dropna(subset=["Data_de_cadastro"])
    .assign(
        grupo_plot=lambda df: df["Grupo_vulnerável"].where(
            df["Grupo_vulnerável"].isin(grupos_destacados),
            "Outros",
        ),
        mes=lambda df: df["Data_de_cadastro"].dt.to_period("M").astype("string"),
    )
    .groupby(["mes", "grupo_plot"], as_index=False)
    .size()
    .rename(columns={"size": "quantidade"})
)

ordem_grupos = [
    "VIOLÊNCIA CONTRA A MULHER",
    "VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+",
    "Outros",
]

fig = px.bar(
    grupo_vulneravel_mes_plot,
    x="mes",
    y="quantidade",
    color="grupo_plot",
    barmode="group",
    category_orders={"grupo_plot": ordem_grupos},
    title="Ocorrências por mês: mulher, população LGBTQIA+ e outros",
    labels={"mes": "Mês", "quantidade": "Quantidade de ocorrências", "grupo_plot": "Grupo"},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

grupo_vulneravel_mes_plot


,mes,grupo_plot,quantidade
0,2025-01,Outros,345915
1,2025-01,VIOLÊNCIA CONTRA A MULHER,31422
2,2025-01,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,4237
3,2025-02,Outros,328278
4,2025-02,VIOLÊNCIA CONTRA A MULHER,27755
5,2025-02,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,2949
6,2025-03,Outros,323116
7,2025-03,VIOLÊNCIA CONTRA A MULHER,31581
8,2025-03,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,3617
9,2025-04,Outros,326795


Contagem de genero das vitimas no mês a mes

In [18]:
genero_vitima_mes_a_mes = (
    disk100_unificado_df.assign(
        Genero_da_vítima=disk100_unificado_df["Genero_da_vítima"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        mes=disk100_unificado_df["Data_de_cadastro"].dt.to_period("M").astype("string"),
    )
    .dropna(subset=["Data_de_cadastro"])
    .groupby(["Genero_da_vítima", "mes"])
    .size()
    .unstack(fill_value=0)
    .sort_index(axis=1)
    .reset_index()
)

genero_vitima_mes_a_mes


mes,Genero_da_vítima,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
0,FEMININO,196692,181567,187052,187709,190729,177087,0,0,0,0,0,0
1,INTERSEXO,298,181,176,170,185,230,0,0,0,0,0,0
2,MASCULINO,158440,151153,146320,147142,153790,139758,0,0,0,0,0,0
3,NÃO INFORMADO,4912,4408,4265,4623,5050,3905,0,0,0,0,0,0
4,NÃO SE APLICA - VÍTIMA COMUNIDADE/FAMÍLIA,639,458,557,503,578,473,0,0,0,0,0,0
5,Não informado,20593,21215,19943,22316,20021,17572,380868,410219,438784,442009,393683,356125
6,nulo,0,0,1,0,0,0,0,0,0,0,0,0


Plot gerado pelo codex do genero das vitimas no mês a mês

In [19]:
genero_vitima_mes_plot = (
    disk100_unificado_df.assign(
        Genero_da_vítima=disk100_unificado_df["Genero_da_vítima"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        mes=disk100_unificado_df["Data_de_cadastro"].dt.to_period("M").astype("string"),
    )
    .dropna(subset=["Data_de_cadastro"])
    .groupby(["mes", "Genero_da_vítima"], as_index=False)
    .size()
    .rename(columns={"size": "quantidade"})
)

fig = px.bar(
    genero_vitima_mes_plot,
    x="mes",
    y="quantidade",
    color="Genero_da_vítima",
    barmode="group",
    title="Ocorrências por mês e gênero da vítima",
    labels={"mes": "Mês", "quantidade": "Quantidade de ocorrências", "Genero_da_vítima": "Gênero da vítima"},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

genero_vitima_mes_plot


,mes,Genero_da_vítima,quantidade
0,2025-01,FEMININO,196692
1,2025-01,INTERSEXO,298
2,2025-01,MASCULINO,158440
3,2025-01,NÃO INFORMADO,4912
4,2025-01,NÃO SE APLICA - VÍTIMA COMUNIDADE/FAMÍLIA,639
5,2025-01,Não informado,20593
6,2025-02,FEMININO,181567
7,2025-02,INTERSEXO,181
8,2025-02,MASCULINO,151153
9,2025-02,NÃO INFORMADO,4408


Filtro de grupo violencia contra grupo vulneravel por uf

In [20]:
grupo_vulneravel_por_uf = (
    disk100_unificado_df.assign(
        Grupo_vulnerável=disk100_unificado_df["Grupo_vulnerável"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        UF=disk100_unificado_df["UF"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
    )
    .groupby(["UF", "Grupo_vulnerável"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
    .reset_index()
)

grupo_vulneravel_por_uf


Grupo_vulnerável,UF,VIOLÊNCIA CONTRA A MULHER,"VIOLÊNCIA CONTRA CIDADÃO, FAMÍLIA OU COMUNIDADE",VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,VIOLÊNCIA CONTRA PESSOA COM DEFICIÊNCIA,VIOLÊNCIA CONTRA PESSOA EM RESTRIÇÃO DE LIBERDADE,VIOLÊNCIA CONTRA PESSOA EM SITUAÇÃO DE RUA,VIOLÊNCIA CONTRA PESSOA IDOSA,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,nulo
0,AC,1183,716,4277,1819,48,14,2715,89,0
1,AL,4698,4504,21736,7564,220,236,9453,605,0
2,AM,7210,7682,41283,16667,683,485,24807,468,0
3,AP,625,699,2781,1222,143,51,1981,51,0
4,ATENDIMENTO INTERROMPIDO,201,88,296,96,8,0,115,46,0
5,BA,24278,20973,89192,40160,1167,1527,48874,4750,0
6,CE,13948,24333,53903,30167,12078,869,39794,1812,0
7,DENUNCIANTE NÃO SOUBE INFORMAR,6511,5496,10514,2587,419,175,2328,1504,0
8,DF,9653,10396,31265,18401,1596,2414,23276,1351,0
9,ES,9561,7070,41141,17829,1151,387,23976,775,0


# editar este abaixo

Plot gerado pelo Codex para vizualização de entradas dos grupos vulneráveis de interesse

In [ ]:
grupos_destacados_uf = {
    "VIOLÊNCIA CONTRA A MULHER",
    "VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+",
}

rotulos_nao_informado = {
    "ATENDIMENTO INTERROMPIDO",
    "DENUNCIANTE NÃO SOUBE INFORMAR",
    "Não informado",
}

grupo_vulneravel_por_uf_plot_ajustado = (
    disk100_unificado_df.assign(
        Grupo_vulnerável=disk100_unificado_df["Grupo_vulnerável"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        UF=disk100_unificado_df["UF"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
    )
    .assign(
        UF=lambda df: df["UF"].replace(list(rotulos_nao_informado), "Não informado"),
        grupo_plot=lambda df: df["Grupo_vulnerável"].where(
            df["Grupo_vulnerável"].isin(grupos_destacados_uf),
            "Outros",
        ),
    )
    .groupby(["UF", "grupo_plot"], as_index=False)
    .size()
    .rename(columns={"size": "quantidade"})
)

ordem_ufs = (
    grupo_vulneravel_por_uf_plot_ajustado
    .groupby("UF", as_index=False)["quantidade"]
    .sum()
    .sort_values("quantidade", ascending=False)["UF"]
    .tolist()
)

ordem_grupos_uf = [
    "VIOLÊNCIA CONTRA A MULHER",
    "VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+",
    "Outros",
]

fig = px.bar(
    grupo_vulneravel_por_uf_plot_ajustado,
    x="UF",
    y="quantidade",
    color="grupo_plot",
    barmode="group",
    category_orders={"grupo_plot": ordem_grupos_uf, "UF": ordem_ufs},
    title="Ocorrências por UF: mulher, população LGBTQIA+ e outros",
    labels={"UF": "UF", "quantidade": "Quantidade de ocorrências", "grupo_plot": "Grupo"},
)
fig.show()

grupo_vulneravel_por_uf_plot_ajustado


,UF,grupo_plot,quantidade
0,AC,Outros,9589
1,AC,VIOLÊNCIA CONTRA A MULHER,1183
2,AC,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,89
3,AL,Outros,43713
4,AL,VIOLÊNCIA CONTRA A MULHER,4698
...,...,...,...
80,SP,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,13565
81,TO,Outros,15519
82,TO,VIOLÊNCIA CONTRA A MULHER,1415
83,TO,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,143


Contagem de denuncias de cada grupo vulnerael por municipio

In [22]:
grupo_vulneravel_por_municipio = (
    disk100_unificado_df.assign(
        Grupo_vulnerável=disk100_unificado_df["Grupo_vulnerável"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        Município=disk100_unificado_df["Município"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
    )
    .groupby(["Município", "Grupo_vulnerável"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
    .reset_index()
)

grupo_vulneravel_por_municipio


Grupo_vulnerável,Município,VIOLÊNCIA CONTRA A MULHER,"VIOLÊNCIA CONTRA CIDADÃO, FAMÍLIA OU COMUNIDADE",VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,VIOLÊNCIA CONTRA PESSOA COM DEFICIÊNCIA,VIOLÊNCIA CONTRA PESSOA EM RESTRIÇÃO DE LIBERDADE,VIOLÊNCIA CONTRA PESSOA EM SITUAÇÃO DE RUA,VIOLÊNCIA CONTRA PESSOA IDOSA,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,nulo
0,1100015 | ALTA FLORESTA D'OESTE,0,0,29,0,0,0,5,0,0
1,1100023 | ARIQUEMES,76,187,591,196,73,0,223,9,0
2,1100031 | CABIXI,0,0,0,0,0,0,10,0,0
3,1100049 | CACOAL,27,74,264,100,0,0,81,1,0
4,1100056 | CEREJEIRAS,11,23,26,26,0,0,45,0,0
...,...,...,...,...,...,...,...,...,...,...
5360,5300108 | BRASÍLIA,9653,10368,31211,18378,1596,2398,23259,1351,0
5361,ATENDIMENTO INTERROMPIDO,247,133,367,147,8,0,159,46,0
5362,DENUNCIANTE NÃO SOUBE INFORMAR,7183,6432,13679,3614,471,197,3817,1642,0
5363,Não informado,1097,633,3385,634,55,36,851,43,0


bosplot de iolências cometidas contra os grupos pro município

In [23]:
grupos_destacados_municipio = {
    "VIOLÊNCIA CONTRA A MULHER",
    "VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+",
}

grupo_vulneravel_por_municipio_plot = (
    disk100_unificado_df.assign(
        Grupo_vulnerável=disk100_unificado_df["Grupo_vulnerável"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        Município=disk100_unificado_df["Município"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
    )
    .assign(
        grupo_plot=lambda df: df["Grupo_vulnerável"].where(
            df["Grupo_vulnerável"].isin(grupos_destacados_municipio),
            "Outros",
        )
    )
    .groupby(["Município", "grupo_plot"], as_index=False)
    .size()
    .rename(columns={"size": "quantidade"})
)

ordem_grupos_municipio = [
    "VIOLÊNCIA CONTRA A MULHER",
    "VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+",
    "Outros",
]

fig = px.box(
    grupo_vulneravel_por_municipio_plot,
    x="grupo_plot",
    y="quantidade",
    category_orders={"grupo_plot": ordem_grupos_municipio},
    points="outliers",
    title="Distribuição das ocorrências por município: mulher, população LGBTQIA+ e outros",
    labels={"grupo_plot": "Grupo", "quantidade": "Quantidade de ocorrências por município"},
)
fig.show()

grupo_vulneravel_por_municipio_plot


,Município,grupo_plot,quantidade
0,1100015 | ALTA FLORESTA D'OESTE,Outros,34
1,1100023 | ARIQUEMES,Outros,1270
2,1100023 | ARIQUEMES,VIOLÊNCIA CONTRA A MULHER,76
3,1100023 | ARIQUEMES,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,9
4,1100031 | CABIXI,Outros,10
...,...,...,...
10210,DENUNCIANTE NÃO SOUBE INFORMAR,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,1642
10211,Não informado,Outros,5594
10212,Não informado,VIOLÊNCIA CONTRA A MULHER,1097
10213,Não informado,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,43


Plot e sort dos dados feito pelo codex para printar em ordem de total de ocorrências

In [25]:
grupo_vulneravel_total = (
    grupo_vulneravel_por_uf
    .drop(columns=["UF"])
    .sum(axis=0)
    .reset_index()
    .rename(columns={"index": "Grupo_vulnerável", 0: "quantidade"})
)

fig = px.pie(
    grupo_vulneravel_total,
    names="Grupo_vulnerável",
    values="quantidade",
    title="Proporção total de ocorrências por grupo vulnerável",
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.show()

grupo_vulneravel_total

,Grupo_vulnerável,quantidade
0,VIOLÊNCIA CONTRA A MULHER,390835
1,"VIOLÊNCIA CONTRA CIDADÃO, FAMÍLIA OU COMUNIDADE",419723
2,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,1819858
3,VIOLÊNCIA CONTRA PESSOA COM DEFICIÊNCIA,757193
4,VIOLÊNCIA CONTRA PESSOA EM RESTRIÇÃO DE LIBERDADE,67235
5,VIOLÊNCIA CONTRA PESSOA EM SITUAÇÃO DE RUA,24851
6,VIOLÊNCIA CONTRA PESSOA IDOSA,1059465
7,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,53238
8,nulo,1


In [26]:
grupo_vulneravel_total_ordenado = grupo_vulneravel_total.sort_values("quantidade", ascending=True)

fig = px.bar(
    grupo_vulneravel_total_ordenado,
    x="quantidade",
    y="Grupo_vulnerável",
    orientation="h",
    title="Total de ocorrências por grupo vulnerável",
    labels={"quantidade": "Quantidade de ocorrências", "Grupo_vulnerável": "Grupo vulnerável"},
)
fig.show()

grupo_vulneravel_total_ordenado

,Grupo_vulnerável,quantidade
8,nulo,1
5,VIOLÊNCIA CONTRA PESSOA EM SITUAÇÃO DE RUA,24851
7,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,53238
4,VIOLÊNCIA CONTRA PESSOA EM RESTRIÇÃO DE LIBERDADE,67235
0,VIOLÊNCIA CONTRA A MULHER,390835
1,"VIOLÊNCIA CONTRA CIDADÃO, FAMÍLIA OU COMUNIDADE",419723
3,VIOLÊNCIA CONTRA PESSOA COM DEFICIÊNCIA,757193
6,VIOLÊNCIA CONTRA PESSOA IDOSA,1059465
2,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,1819858


In [27]:
grupo_vulneravel_total_percentual = (
    grupo_vulneravel_total
    .assign(
        porcentagem_do_total=lambda df: (df["quantidade"] / df["quantidade"].sum() * 100).round(2)
    )
    .sort_values("quantidade", ascending=False)
    .reset_index(drop=True)
)

grupo_vulneravel_total_percentual


,Grupo_vulnerável,quantidade,porcentagem_do_total
0,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,1819858,39.63
1,VIOLÊNCIA CONTRA PESSOA IDOSA,1059465,23.07
2,VIOLÊNCIA CONTRA PESSOA COM DEFICIÊNCIA,757193,16.49
3,"VIOLÊNCIA CONTRA CIDADÃO, FAMÍLIA OU COMUNIDADE",419723,9.14
4,VIOLÊNCIA CONTRA A MULHER,390835,8.51
5,VIOLÊNCIA CONTRA PESSOA EM RESTRIÇÃO DE LIBERDADE,67235,1.46
6,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,53238,1.16
7,VIOLÊNCIA CONTRA PESSOA EM SITUAÇÃO DE RUA,24851,0.54
8,nulo,1,0.00
